## 9장 2강 : 비동기 함수 Promise<T>와 에러 모델링

### 1. Promise와 async/await의 타입 설계
`Promise<T>`는 미래에 완료될 비동기 작업의 결과 데이터 타입 T를 지정하는 제네릭 타입입니다.

#### 1.1 Promise의 기본 구조와 반환 타입
- `Promise<T>` 개념
> 비동기 작업이 성공했을 때 반환할 데이터의 타입을 제네릭 <T>로 정의합니다.

- 반환 타입 명시
> 비동기 함수는 내부에서 값을 리턴하더라도 반환 타입이 항상 `Promise<T>` 형태가 됩니다.

In [1]:
// 사용자 데이터 인터페이스를 정의합니다.
interface User {
  id: number; // 사용자 고유 식별자 숫자
  name: string; // 사용자 이름 문자열
  email: string; // 사용자 이메일 문자열
}

// Promise<User>를 반환하는 비동기 함수 예시입니다.
function fetchUserData(): Promise<User> {
  return new Promise((resolve) => {
    // 1초 뒤에 User 객체를 반환하는 비동기 작업 실행
    setTimeout(() => {
      resolve({
        id: 1,
        name: "김철수",
        email: "chulsoo@example.com"
      });
    }, 1000);
  });
}

#### 1.2 async/await 함수와 Top-level await
- async 키워드
> 함수 앞에 async를 붙이면 해당 함수는 무조건 Promise를 반환합니다.
- await 키워드
> Promise의 이행을 기다리고, 내부의 순수 데이터 T를 추출합니다.
- Top-level await
> tslab 주피터 노트북 환경에서는 async 함수로 감싸지 않고도 셀 최상위에서 await를 즉시 사용할 수 있습니다.

In [2]:
// async 함수는 리턴값에 대해 자동으로 Promise 포장을 적용합니다.
async function getUserName(userId: number): Promise<string> {
  // 실제 백엔드 연동을 가정한 모의 비동기 처리
  return `사용자_${userId}`;
}

// tslab 환경에서는 최상위 셀에서 await를 바로 실행할 수 있습니다.
const userName = await getUserName(101);
// 추출된 결과는 Promise<string>이 아닌 string 타입입니다.
console.log(`조회된 사용자 이름: ${userName}`);

조회된 사용자 이름: 사용자_101


### 2. 백엔드 에러 응답 모델링 및 unknown 에러 처리
fetch API는 HTTP 에러 상태(404, 500)에서도 reject 되지 않으므로, response.ok 검증과 커스텀 에러 모델링이 필요합니다.

#### 2.1 HTTP 에러 응답 클래스 및 인터페이스 설계
- fetch 특성
> 네트워크 오류 시에만 catch로 이동하며, HTTP 404/500 에러는 response.ok가 false로 설정됩니다.

- 커스텀 에러 모델링
> HTTP 에러 상태 코드와 메시지를 담을 수 있는 커스텀 에러 클래스를 작성합니다.

In [3]:
// 백엔드 API의 공통 에러 응답 구조를 정의하는 인터페이스입니다.
interface ApiErrorResponse {
  statusCode: number; // HTTP 상태 코드 (예: 400, 404, 500)
  message: string; // 에러 상세 메시지
  errorCode: string; // 시스템 내부 에러 식별 코드
}

// 명확한 에러 처리를 위한 커스텀 HTTP 에러 클래스입니다.
class HttpError extends Error {
  statusCode: number;
  errorCode: string;

  constructor(statusCode: number, message: string, errorCode: string) {
    super(message);
    this.statusCode = statusCode;
    this.errorCode = errorCode;
    this.name = 'HttpError';
  }
}

#### 2.2 catch 블록의 unknown 타입 가드와 안전한 에러 처리
- catch(error)의 기본 타입
> TypeScript에서 catch 절의 에러 객체는 안전성을 위해 unknown 타입으로 지정됩니다.
- 타입 가드 사용
> instanceof 연산자를 활용하여 HttpError 또는 일반 Error 객체인지 검증한 뒤 안전하게 속성에 접근합니다.

In [4]:
// 에러 객체를 검증하고 처리하는 공통 함수입니다.
function handleServerError(error: unknown): void {
  // 사용자 정의 HttpError 인스턴스인지 검증
  if (error instanceof HttpError) {
    console.log(`[HTTP 에러 발생] 상태코드: ${error.statusCode}, 코드: ${error.errorCode}, 메시지: ${error.message}`);
  } 
  // 일반 JavaScript Error 인스턴스인지 검증
  else if (error instanceof Error) {
    console.log(`[일반 에러 발생] ${error.message}`);
  } 
  // 기타 알 수 없는 에러 처리
  else {
    console.log("알 수 없는 에러가 발생했습니다.", error);
  }
}

### 3. fetch 기반 비동기 API 통신 및 예외 처리 시스템


In [5]:
// API 성공 응답 데이터 구조 정의
interface TodoItem {
  userId: number; // 작성자 ID
  id: number; // 할 일 ID
  title: string; // 할 일 제목
  completed: boolean; // 완료 여부
}

// fetch 기반 비동기 데이터 조회 함수 작성
async function fetchTodoById(todoId: number): Promise<TodoItem> {
  // fetch API를 사용해 데이터를 요청합니다.
  const response = await fetch(`https://jsonplaceholder.typicode.com/todos/${todoId}`);

  // HTTP 응답 상태가 200~299 범위가 아닌 경우 예외를 발생시킵니다.
  if (!response.ok) {
    throw new HttpError(
      response.status,
      `요청 실패: 상태 코드 ${response.status}`,
      "FETCH_ERROR"
    );
  }

  // response.json() 결과는 알 수 없는 타입이므로 'as TodoItem'으로 타입 단언을 수행합니다.
  const data = (await response.json()) as TodoItem;
  return data;
}

// 정상 동작 케이스 테스트
try {
  const todoData = await fetchTodoById(1);
  console.log("=== 성공적으로 데이터를 가져왔습니다 ===");
  console.log(`제목: ${todoData.title}`);
  console.log(`완료 여부: ${todoData.completed}`);
} catch (error: unknown) {
  handleServerError(error);
}

// 에러 발생 케이스 테스트 (잘못된 URL 요청 시뮬레이션)
try {
  console.log("\n=== 예외 처리 테스트 실행 ===");
  const invalidUrl = "https://jsonplaceholder.typicode.com/invalid-endpoint-12345";
  const response = await fetch(invalidUrl);

  // 404 에러가 발생하면 response.ok가 false가 됩니다.
  if (!response.ok) {
    throw new HttpError(
      response.status,
      "요청한 API 엔드포인트를 찾을 수 없습니다.",
      "NOT_FOUND"
    );
  }
} catch (error: unknown) {
  handleServerError(error);
}

=== 성공적으로 데이터를 가져왔습니다 ===
제목: delectus aut autem
완료 여부: false

=== 예외 처리 테스트 실행 ===
[HTTP 에러 발생] 상태코드: 404, 코드: NOT_FOUND, 메시지: 요청한 API 엔드포인트를 찾을 수 없습니다.
